# 01. Exploratory Data Analysis & Baseline Model Evaluation

This notebook demonstrates:
1. Ingestion of news datasets.
2. Text length and lexical feature distributions (uppercase ratio, exclamation counts).
3. TF-IDF feature importance analysis.
4. Calibrated threshold evaluation (Fake / Real / Uncertain).

In [ ]:
from fake_news_detector.data.loader import create_synthetic_news_dataset
from fake_news_detector.data.preprocessor import TextPreprocessor
from fake_news_detector.models.baseline import TfidfLogisticRegressionClassifier
from fake_news_detector.models.decision import DecisionEngine

In [ ]:
# 1. Ingest Dataset
df = create_synthetic_news_dataset(num_samples=30)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# 2. Preprocessing
preprocessor = TextPreprocessor()
df["cleaned_text"] = preprocessor.transform(df["text"].tolist())
df[["title", "cleaned_text", "label"]].head()

In [ ]:
# 3. Fit Baseline Model
model = TfidfLogisticRegressionClassifier(max_features=1000)
model.fit(df["cleaned_text"].tolist(), df["label"].tolist())
print("Baseline model trained successfully.")

In [ ]:
# 4. Evaluate Decision Thresholds
engine = DecisionEngine(fake_threshold_high=0.70, fake_threshold_low=0.30)
sample_probs = model.predict_proba(df["cleaned_text"].tolist()[:5])
for i, p in enumerate(sample_probs):
    res = engine.evaluate(p[1])
    print(
        f"Sample {i + 1} -> Verdict: {res.verdict}, Conf: {res.confidence_score}, P(Fake): {res.prob_fake}"
    )